### Checking Full File Chunking

In [1]:
import json
from pathlib import Path
from src.ingestion.chunker import chunk_document

INPUT_FILE = Path(r"C:\Users\SHREY\Desktop\t1d_bot\dataset\processed\chapters\ISPAD-English-2022\Ch12-PediatricDiabetes.json")
OUTPUT_FILE = Path("dataset/processed/chunks_testing/ISPAD-English-2022/Ch12-PediatricDiabetes.json")

# Ensure output folder exists
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    chapters = json.load(f)

# Normalize to list
if isinstance(chapters, dict):
    chapters = [chapters]

print(f"[INFO] Input chapters: {len(chapters)} from {INPUT_FILE.name}")

chunks = chunk_document(chapters)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"[DONE] Saved {len(chunks)} chunks -> {OUTPUT_FILE}")

[INFO] Input chapters: 27 from Ch12-PediatricDiabetes.json
[SKIP] Skipping chapter (substring match): I S P A D G U I D E L I N E S ISPAD Clinical Practice Consensus Guidelines 2022: Assessment and management of hypoglycemia in children and adolescents with diabetes
[INFO] Chunking chapter 2: K E Y W O R D S : glucagon, hypoglycemia, impaired awareness of hypoglycemia
[INFO] Chunking chapter 3: 1 | WHAT IS NEW OR DIFFERENT?
[INFO] Chunking chapter 4: 2 | EXECUTIVE SUMMARY AND RECOMMENDATIONS
[INFO] Chunking chapter 5: 3 | INTRODUCTION
[INFO] Chunking chapter 6: 4 | DEFINITION AND INCIDENCE
[INFO] Chunking chapter 7: 4.1 | Definition
[INFO] Chunking chapter 8: 4.2 | Incidence
[INFO] Chunking chapter 9: 5 | MORBIDITY AND MORTALITY WITH HYPOGLYCEMIA
[INFO] Chunking chapter 10: 5.1 | Mortality
[INFO] Chunking chapter 11: TABLE 1
[INFO] Chunking chapter 12: 5.2 | Morbidity
[INFO] Chunking chapter 13: TABLE 2
[INFO] Chunking chapter 14: 6 | SIGNS AND SYMPTOMS
[INFO] Chunking chapter 15: 7 | 

ReadError: [WinError 10053] An established connection was aborted by the software in your host machine

### Comments on Semantic Chunking Quality (as evaluated by the LLM GPT-4.1)

#### GENERAL CHUNKS

Summary:
The semantic chunking is well-aligned with document structure and meaning. Chunks are coherent, contextually complete, and respect logical boundaries. Only minor tuning may be needed depending on your downstream use case.

#### TABULAR CHUNKS

Conclusion:
The chunking for tables is high quality for semantic and contextual retrieval, but not for structured data extraction. If you need tables as structured data, a dedicated table extraction step would be required. For reading, search, or QA tasks, the current chunking is effective and context-preserving.

### Final Run

In [5]:
import json
from pathlib import Path
import src.ingestion.chunker
import importlib
importlib.reload(src.ingestion.chunker)
from src.ingestion.chunker import chunk_document_resumable


INPUT_DIR = Path(r"C:\Users\SHREY\Desktop\t1d_bot\dataset\processed\chapters\ISPAD-English-2022")
OUTPUT_DIR = Path("dataset/processed/chunks/ISPAD-English-2022")
CHECKPOINT_DIR = OUTPUT_DIR / "_checkpoints"

# Ensure output folders exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

input_files = sorted(INPUT_DIR.glob("*.json"))
print(f"[INFO] Found {len(input_files)} input files in {INPUT_DIR}")

for input_file in input_files:
    output_file = OUTPUT_DIR / input_file.name  # keep exact same filename
    checkpoint_file = CHECKPOINT_DIR / f"{input_file.stem}.checkpoint.json"

    if output_file.exists():
        print(f"[SKIP] {output_file} already exists, skipping.")
        continue

    with open(input_file, "r", encoding="utf-8") as f:
        chapters = json.load(f)

    # Normalize to list
    if isinstance(chapters, dict):
        chapters = [chapters]

    print(f"[INFO] Processing {input_file.name} | chapters: {len(chapters)}")

    chunks = chunk_document_resumable(chapters, checkpoint_file)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    if checkpoint_file.exists():
        checkpoint_file.unlink()
        print(f"[CLEANUP] Removed checkpoint {checkpoint_file}")

    print(f"[DONE] Saved {len(chunks)} chunks -> {output_file}")

[INFO] Found 19 input files in C:\Users\SHREY\Desktop\t1d_bot\dataset\processed\chapters\ISPAD-English-2022
[SKIP] dataset\processed\chunks\ISPAD-English-2022\Ch1-DefinitionEpidemiol.json already exists, skipping.
[SKIP] dataset\processed\chunks\ISPAD-English-2022\Ch10PediatricDiabetes-2.json already exists, skipping.
[SKIP] dataset\processed\chunks\ISPAD-English-2022\Ch11PediatricDiabetes.json already exists, skipping.
[SKIP] dataset\processed\chunks\ISPAD-English-2022\Ch12-PediatricDiabetes.json already exists, skipping.
[SKIP] dataset\processed\chunks\ISPAD-English-2022\Ch13PediatricDiabetes-2.json already exists, skipping.
[SKIP] dataset\processed\chunks\ISPAD-English-2022\Ch14-PediatricDiabetes.json already exists, skipping.
[SKIP] dataset\processed\chunks\ISPAD-English-2022\Ch15PediatricDiabetes-2.json already exists, skipping.
[SKIP] dataset\processed\chunks\ISPAD-English-2022\Ch16-Technologyglucosem.json already exists, skipping.
[SKIP] dataset\processed\chunks\ISPAD-English-20

ValueError: LLM output likely truncated

Summary:
The semantic chunking in this file is of high quality. Chunks are logically segmented, thematically coherent, and appropriately sized for downstream NLP or retrieval tasks. The process preserves semantic integrity and aligns well with the document’s structure. Minor improvements could be made in chunk naming for human readers, but overall, the chunking is robust and effective.

<b> Issue with Chap 22, to be diagnosed later <b>